### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

### Clean Up Data

<small>Read the data and convert into dataframe</small>
<small></small>

In [ ]:
pip install kagglehub

In [ ]:
from os import read
# read data from csv

import kagglehub

# Download latest version
path = kagglehub.dataset_download("ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training")

df = pd.read_csv(path + '/dirty_cafe_sales.csv')
print("Data loaded successfully.")

<small>Check the data set type using info</small>

In [ ]:
df.info()

<small>There are some null values in the dataset that need to be handled.</small>

<small>Check how the actual values look (overview)</small>

In [ ]:
df.head(40)

<small>Check unique values to sort out Erroneous data types</small>

<small>Based on the data, Erroneous data types are: ERROR, NaN, UNKNOWN</small>

In [ ]:
# check column unique values 
print(df.nunique())

# get for the Item, Price Per Unit, Payment Method, Location
unique_items = df['Item'].unique().tolist()
unique_prices = df['Price Per Unit'].unique().tolist()
unique_payment_methods = df['Payment Method'].unique().tolist()
unique_locations = df['Location'].unique().tolist()


print("Unique Items:", unique_items)
print("Unique Prices:", unique_prices)
print("Unique Payment Methods:", unique_payment_methods)   
print("Unique Locations:", unique_locations)

# take note of erroneous data types
error_values = ["ERROR", "UNKNOWN", "NAN"]

<small>Check for null values per column</small>

In [ ]:
print("Null Values in Each Column:")
print(df.isnull().sum())

<small>Check the data types</small>

In [ ]:
df.info()

<small>Convert to appropriate data type, apply coerce on errors to change to NaN or NaT</small>

In [ ]:
# convert to correct data types

# numerical columns
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')

df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')
df['Total Price'] = df['Quantity'] * df['Price Per Unit']

# categorical columns
df['Item'] = df['Item'].astype('string')
df['Payment Method'] = df['Payment Method'].astype('string')
df['Location'] = df['Location'].astype('string')

# datetime column
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')


<small>Recheck newly casted data types</small>

In [ ]:
df.info()

<small>Create a copy of the data frame. Make a helper function that converts values from the error_values list taken to a panda (pd.NA) or null value type in pandas. apply it on the copied dataframe, with axis set to "1" to apply to the rows</small>

In [ ]:

df_clean = df.copy()

def clean_data(row): 
    for col in df_clean.columns:
        if isinstance(row[col], str):
            if row[col].upper() in error_values:
                row[col] = pd.NA
    return row

df_clean = df_clean.apply(clean_data, axis=1)


<small>Check actual null values are reflected in rows</small>

In [ ]:
df_clean.head(40)

<small>Recheck the number of nulls in the columns to see that they increased</small>

In [ ]:
df_clean.isna().sum()

<small>Use different approaches to fill pd.NA types with appropriate values. By default, use Medians for numerical values, Mode for categorical. Total Price is a column calculated using other columns instead</small>

In [ ]:
# First create a menu view table
menu_data = df_clean.groupby('Item')['Price Per Unit'].first().sort_values(ascending=False)
print(menu_data)

In [ ]:
# fill data based on type, median or mode

# based on the data, and information on the data set
# We can use that information to infer missing values.
# In this case, we can make a dict of a corresponding item per price per unit
item_price_dict = menu_data.to_dict()

print("Menu Data:")
print(item_price_dict)

"""
Numerical Columns
"""
# Item
# Based on the data, get the item where the price per unit is the most frequent
# if any nulls remain, fill in with the mode
df_clean['Item']= df_clean['Item'].fillna(df_clean['Price Per Unit'].map({v: k for k, v in item_price_dict.items()}))
df_clean['Item']= df_clean['Item'].fillna(df_clean['Item'].mode()[0])

# Price Per Unit
# Based on the data, fill in with the calculated total spent divided by quantity
# if any nulls remain, fill in with the item price
df_clean['Price Per Unit']= df_clean['Price Per Unit'].fillna((df_clean['Total Spent'] / df_clean['Quantity']))
df_clean['Price Per Unit']= df_clean['Price Per Unit'].fillna(df_clean['Item'].map(item_price_dict))

# Quantity
# Based on the data, fill in with the calculated total spent divided by price per unit
df_clean['Quantity']= df_clean['Quantity'].fillna((df_clean['Total Spent'] / df_clean['Price Per Unit']))

# Total Spent
# Based on the data, fill in with the calculated quantity multiplied by price per unit
df_clean['Total Spent']= df_clean['Total Spent'].fillna(df_clean['Quantity'] * df_clean['Price Per Unit'])

# if any null remains in numerical columns, fill with median or mean as appropriate
# in this case, we use median to avoid skew from outliers
df_clean['Price Per Unit']= df_clean['Price Per Unit'].fillna(df_clean['Price Per Unit'].median())
df_clean['Quantity']= df_clean['Quantity'].fillna(df_clean['Quantity'].median())
df_clean['Total Spent']= df_clean['Total Spent'].fillna(df_clean['Total Spent'].median())

# Case Quantity to integer
df_clean['Quantity'] = df_clean['Quantity'].map(lambda x: int(x))


"""
Categorical Columns
"""
# Payment Method
# Fill with mode
df_clean['Payment Method']= df_clean['Payment Method'].fillna(df_clean['Payment Method'].mode()[0],)

# Location
# Fill with mode
df_clean['Location']= df_clean['Location'].fillna(df_clean['Location'].mode()[0])

# Transaction Date
# In order to differentiate missing dates, fill with a default date, in this case 1990-01-01
default_time = pd.Timestamp("1990-01-01 00:00:00")
df_clean['Transaction Date']= df_clean['Transaction Date'].fillna(default_time)

# Total Price
# At this point, quantity and price per unit should be filled, so recalculate
df_clean['Total Price'] = df_clean['Quantity'] * df_clean['Price Per Unit']

print("Succesfully turned null values in non-null.")


<small>Recheck and confirm that there are no null values present in each column and that the Juice items have the correct price per unit.</small>

In [ ]:
df_clean.isna().sum()

<small> **Extra Step:** Based on data observations, some rows contain inconsistent item-price mappings. Specifically, items labeled as "Juice" may have incorrect price values. To correct this, we'll use the `item_price_dict` to reverse-map prices back to their correct item names, ensuring consistency between items and their prices.</small>

In [ ]:
# check unique item-price pairs to ensure consistency
df_clean[['Item', 'Price Per Unit']].drop_duplicates()


<small>Handle the Juice items with incorrect Price Per Unit by using the menu data to correct them based on the price</small>

In [ ]:
# Find all rows where Item is Juice but Price Per Unit is not the correct juice price
# and update the Item based on the Price Per Unit using the reverse mapping of item_price_dict

# Fix Juice items that has incorrect price per unit

juice_price = item_price_dict.get('Juice', None)

price_to_item_map = {v: k for k, v in item_price_dict.items()}
print("Price to Item Map:")
print(price_to_item_map)

df_clean.loc[
    (df_clean['Item'] == 'Juice') &
    (df_clean['Price Per Unit'].notna()) &
    (df_clean['Price Per Unit'] != juice_price),
    'Item'
] = df_clean.loc[
    (df_clean['Item'] == 'Juice') &
    (df_clean['Price Per Unit'].notna()) &
    (df_clean['Price Per Unit'] != juice_price),
    'Price Per Unit'
].map(price_to_item_map)

print("Data cleaning completed successfully for Juice items.")


<small>Now we can recheck and confirm that there are no null values present in each column and that the Juice items have the correct price per unit.</small>

In [ ]:
df_clean[['Item', 'Price Per Unit']].drop_duplicates()

<small>Ensure Assumptions</small>

In [ ]:
# Check Unique Values After Cleaning
clean_unique_items = df_clean['Item'].unique().tolist()
clean_unique_prices = df_clean['Price Per Unit'].unique().tolist()
clean_unique_payment_methods = df_clean['Payment Method'].unique().tolist()
clean_unique_locations = df_clean['Location'].unique().tolist()
print("Clean Unique Items:", clean_unique_items)
print("Clean Unique Prices:", clean_unique_prices)
print("Clean Unique Payment Methods:", clean_unique_payment_methods)
print("Clean Unique Locations:", clean_unique_locations)

<small>Visualize the cleaned data</small>

In [ ]:
df_clean.head(40)

### Feature Engineering

<small>Add a feature to categorize an item to either a 'food' or 'drink' depending on the unique item value found in the item column </small>

In [ ]:
clean_unique_items = df_clean['Item'].unique().tolist()
print("Clean Unique Items:", clean_unique_items)

<small>Based on the data of the unique values, categorize them into food and drink</small>

In [ ]:
drinks = ['coffee', 'smoothie', 'juice', 'tea']
food = ['cake', 'cookie', 'salad', 'sandwich']

<small>Make a new column that lists the item as either a food or drink depending on the defined lists above</small>

In [ ]:
df_clean["Item Type"] = df_clean["Item"].apply(
    lambda x: "Drink" if pd.notna(x) and x.lower() in drinks else "Food"
)

<small>Check if the new column is added</small>

In [ ]:
df_clean.head()

<small>Add more columns to prepare for analysis</small>

In [ ]:
# transaction day
df_clean['Transaction Day'] = df_clean['Transaction Date'].dt.day_name()

# transaction month 
df_clean['Transaction Month'] = df_clean['Transaction Date'].dt.month_name()

# transaction year
df_clean['Transaction Year'] = df_clean['Transaction Date'].dt.year

print('Successfully extracted added datetime features.')


### Analysis

<small>Total Sold Per Item, and total earnings</small>

In [ ]:
total_sold_per_item = df_clean.groupby('Item')[['Quantity', 'Total Price']].sum().reset_index().sort_values(by='Quantity', ascending=False)

# Convert Quantity to integer
total_sold_per_item['Quantity'] = total_sold_per_item['Quantity'].map(lambda x: int(x))

total_sold_per_item

<small>Most to least sold item per day</small>

In [ ]:
popular_items_per_day = df_clean.groupby(['Transaction Day', 'Item'])[['Quantity', 'Total Price']].sum().reset_index()
popular_items_per_day = popular_items_per_day.sort_values(['Transaction Day', 'Quantity'], ascending=[True, False])

# Convert Quantity to integer
popular_items_per_day['Quantity'] = popular_items_per_day['Quantity'].map(lambda x: int(x))

# Turn Total Price into currency format
popular_items_per_day['Total Price'] = popular_items_per_day['Total Price'].map(lambda x: f"${x:,.2f}")

popular_items_per_day

<small>Most to least sold item per month </small>

In [ ]:
popular_items_per_month = df_clean.groupby(['Transaction Month', 'Item'])[['Quantity', 'Total Price']].sum().reset_index()
popular_items_per_month = popular_items_per_month.sort_values(['Transaction Month', 'Quantity'], ascending=[True, False])

# Convert Quantity to integer
popular_items_per_month['Quantity'] = popular_items_per_month['Quantity'].map(lambda x: int(x))

# Turn Total Price into currency format
popular_items_per_month['Total Price'] = popular_items_per_month['Total Price'].map(lambda x: f"${x:,.2f}")

popular_items_per_month

<small>Most to least sold item per year</small>

In [ ]:
popular_items_per_year = df_clean.groupby(['Transaction Year', 'Item'])[['Quantity', 'Total Price']].sum().reset_index()
popular_items_per_year = popular_items_per_year.sort_values(['Transaction Year', 'Quantity'], ascending=[True, False])

# Convert Quantity to integer
popular_items_per_year['Quantity'] = popular_items_per_year['Quantity'].map(lambda x: int(x))

# Turn Total Price into currency format
popular_items_per_year['Total Price'] = popular_items_per_year['Total Price'].map(lambda x: f"${x:,.2f}")

popular_items_per_year

<small>Most Popular Location Types</small>

In [ ]:
most_popular_location = df_clean.groupby('Location')[['Quantity', 'Total Price']].sum().reset_index().sort_values(by='Quantity', ascending=False)
print(most_popular_location)

print(df_clean["Location"].mode()[0])

<small>Most Popular Payment Methods</small>

In [ ]:
most_popular_payment_method = df_clean.groupby('Payment Method')[['Quantity', 'Total Price']].sum().reset_index().sort_values(by='Quantity', ascending=False)
print(most_popular_payment_method)

print(df_clean["Payment Method"].mode()[0])

### Visualization

<small>Visualize a menu</small>

In [ ]:
menu_view = df_clean[['Item', 'Price Per Unit', 'Item Type']].drop_duplicates().sort_values(by='Price Per Unit')

menu_view['Price Per Unit'] = menu_view['Price Per Unit'].map('${:,.2f}'.format)

fig, axes = plt.subplots(figsize=(6,4))

axes.axis('tight')
axes.axis('off')

table = axes.table(
    cellText=menu_view.values,
    colLabels=menu_view.columns,
    loc='center',
    cellLoc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(12)

<small>Total Sold Per Item, and total earnings (Visualization)</small>

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Define manual colors
color_quantity = "skyblue"
color_price = "salmon"

# Barplot for total quantity sold
sns.barplot(
    data=total_sold_per_item,
    x="Item", y="Quantity",
    ax=axes[0], color=color_quantity
)
axes[0].set_title("Total Sold Per Item")
axes[0].set_xlabel("Item")
axes[0].set_ylabel("Quantity")
axes[0].tick_params(axis="x", rotation=45)

# Barplot for total earnings
sns.barplot(
    data=total_sold_per_item,
    x="Item", y="Total Price",
    ax=axes[1], color=color_price
)
axes[1].set_title("Total Earnings Per Item")
axes[1].set_xlabel("Item")
axes[1].set_ylabel("Total Price")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


<small>Most to least sold item per day(Visualization)</small>

In [ ]:
pivot = popular_items_per_day.pivot(index="Item", columns="Transaction Day", values="Quantity")
pivot = pivot.astype(int)   

plt.figure(figsize=(10,6))
sns.heatmap(pivot, annot=True, fmt="d", cmap="Blues")
plt.title("Most to Least Sold Items per Day (Quantity)")
plt.ylabel("Item")
plt.xlabel("Transaction Day")
plt.show()


<small>Most to least sold item per month(Visualization) </small>

In [ ]:

pivot_month = popular_items_per_month.pivot(
    index="Item", 
    columns="Transaction Month", 
    values="Quantity"
)


pivot_month = pivot_month.fillna(0).astype(int)

plt.figure(figsize=(12, 7))
sns.heatmap(
    pivot_month, 
    annot=True, fmt="d", cmap="Blues"
)
plt.title("Most to Least Sold Items per Month (Quantity)")
plt.ylabel("Item")
plt.xlabel("Transaction Month")
plt.show()


<small>Most to least sold item per year(Visualization) </small>

In [ ]:

pivot_year = popular_items_per_year.pivot(
    index="Item",
    columns="Transaction Year",
    values="Quantity"
)

pivot_year = pivot_year.fillna(0).astype(int)

plt.figure(figsize=(12, 7))
sns.heatmap(
    pivot_year,
    annot=True, fmt="d", cmap="Blues"
)
plt.title("Most to Least Sold Items per Year (Quantity)")
plt.ylabel("Item")
plt.xlabel("Transaction Year")
plt.show()


<small>Most Popular Location Types(Visualization)</small>

In [ ]:
plt.figure(figsize=(8,8))
plt.pie(
    most_popular_location["Quantity"],
    labels=most_popular_location["Location"],
    autopct="%1.1f%%",
    startangle=140,
    colors=plt.cm.Paired.colors
)
plt.title("Share of Total Quantity Sold by Location")
plt.show()

# Pie chart for Total Price (earnings) per location
plt.figure(figsize=(8,8))
plt.pie(
    most_popular_location["Total Price"],
    labels=most_popular_location["Location"],
    autopct="%1.1f%%",
    startangle=140,
    colors=plt.cm.Set3.colors
)
plt.title("Share of Total Earnings by Location")
plt.show()


In [ ]:
plt.figure(figsize=(8,8))
plt.pie(
    most_popular_payment_method["Quantity"],
    labels=most_popular_payment_method["Payment Method"],
    autopct="%1.1f%%",
    startangle=140,
    colors=plt.cm.Paired.colors
)
plt.title("Share of Total Quantity Sold by Payment Method")
plt.show()

# Pie chart for Total Price (earnings) per payment method
plt.figure(figsize=(8,8))
plt.pie(
    most_popular_payment_method["Total Price"],
    labels=most_popular_payment_method["Payment Method"],
    autopct="%1.1f%%",
    startangle=140,
    colors=plt.cm.Set3.colors
)
plt.title("Share of Total Earnings by Payment Method")
plt.show()
